# 04_模型记忆管理系统

## LangChain 的记忆管理

In [1]:
from langchain_deepseek import ChatDeepSeek

# 准备两种 DeepSeek 模型，后续需要进行模型切换
api_key = ''
base_url = 'https://api.deepseek.com/v1'

basic_model = ChatDeepSeek(model="deepseek-chat", api_key=api_key, base_url=base_url)
reasoner_model = ChatDeepSeek(model="deepseek-reasoner", api_key=api_key, base_url=base_url)

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    model = basic_model, 
    checkpointer=InMemorySaver(),  
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "我喜欢吃苹果和香蕉"}]},
    {"configurable": {"thread_id": "1"}},  
)

In [3]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "我周末准备去跑步"}]},
    {"configurable": {"thread_id": "1"}},  
)
response

{'messages': [HumanMessage(content='我喜欢吃苹果和香蕉', additional_kwargs={}, response_metadata={}, id='84f03bd4-9e20-4d2d-8c33-6729d0b7089d'),
  AIMessage(content='听起来你很喜欢水果呢！苹果和香蕉都是非常健康的选择——苹果富含纤维和维生素C，香蕉则能快速补充能量，还含有钾元素。  \n你平时喜欢怎么吃它们呢？比如直接生吃、做成水果沙拉，还是尝试过烤苹果、香蕉奶昔之类的做法？如果有兴趣的话，我还可以分享一些简单的创意食谱哦！ 😊', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 9, 'total_tokens': 87, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '6287092f-e7e4-4976-9b4c-ffb49a66ec1b', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef8fc-081a-7b40-bfca-4491aa192b93-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 78, 'total_tokens': 87, 'input_token_

## 实现推理过程中模型的动态选择

In [23]:
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@wrap_model_call
def dynamic_deepseek_routing(request: ModelRequest, handler) -> ModelResponse:
    """
    根据对话复杂度动态选择 DeepSeek 模型：
    - 复杂：deepseek-reasoner
    - 简单：deepseek-chat
    """

    print("===== 路由中间件被调用 =====")
    # 从request中获取用户最近一轮对话
    last_user = ''
    last_len = len(last_user)
    messages = request.state.get("messages", [])
    for m in reversed(messages):
        if isinstance(m, HumanMessage):
            if isinstance(m.content, str):
                last_user = m.content
                last_len = len(last_user)
                break
    
    # 一些复杂任务关键词（可自己按需扩充）
    hard_keywords = ("证明", "推导", "严谨", "规划", "多步骤", "chain of thought",
                     "step-by-step", "reason step by step", "数学", "逻辑证明", "约束求解")

    # 如果用户输入过程或者出现推理关键词，就使用推理模型
    is_hard = (
        last_len > 50 or
        any(kw.lower() in last_user.lower() for kw in hard_keywords)
    )

    # 选择模型
    ues_model = reasoner_model if is_hard else basic_model

    print("用户问题:", last_user)
    print("长度:", last_len)
    print("is_hard:", is_hard)
    print(
    "最终模型:",
    reasoner_model.model_name if is_hard else basic_model.model_name)
    # 调用被包裹的下游（真正的模型调用）
    return handler(request.override(model=ues_model))

上述ModelRequest是输入model前必要的参数。handle是把当前权限交给下一个节点执行，可以理解为调用模型运行，所以得到的结果就是ModelResponse

request：当前模型调用请求

request.override(...)：生成一个新的请求

handler(...)：继续执行下游模型调用

ModelResponse：下游返回的模型结果

In [24]:
# 创建 Agent（默认用 chat，但运行时会被中间件按需替换）
agent = create_agent(
    model=basic_model,
    middleware=[dynamic_deepseek_routing],
    checkpointer=InMemorySaver()
)

In [25]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "我喜欢苹果和香蕉"}]},
    {"configurable": {"thread_id": "1"}},
)

===== 路由中间件被调用 =====
用户问题: 我喜欢苹果和香蕉
长度: 8
is_hard: False
最终模型: deepseek-chat


In [26]:
# 复杂问题，用推理模型
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你帮我推导证明一下勾股定理"}]},
    {"configurable": {"thread_id": "1"}},
)

===== 路由中间件被调用 =====
用户问题: 你帮我推导证明一下勾股定理
长度: 13
is_hard: True
最终模型: deepseek-reasoner


In [27]:
response

{'messages': [HumanMessage(content='我喜欢苹果和香蕉', additional_kwargs={}, response_metadata={}, id='b8b63bf1-24e7-4b61-a378-24a2efd605a1'),
  AIMessage(content='听起来你是一个喜欢水果的人！苹果和香蕉都是非常健康的选择：苹果富含纤维和维生素C，香蕉则提供丰富的钾和快速能量。你更常吃哪种呢？或者有特别喜欢的吃法吗？比如苹果派、香蕉奶昔，或者直接当零食？ 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 8, 'total_tokens': 69, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'f8393f14-6e50-478b-99bc-32443073ffb1', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ef90f-08c2-7ab1-846a-872c66dc8cfb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 61, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}, 'output_

## 推理前实现记忆修剪

In [28]:
from langchain.agents import AgentState
from langchain.agents.middleware import before_model
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """在模型调用前修剪消息历史，只保留前1+后3条。"""
    messages = state["messages"]

    # 只有超过 4 条消息才裁剪
    if len(messages) <= 4:
        return None

    # 保留首条 System 提示 + 最近3条
    first_msg = messages[0]
    new_messages = [first_msg] + messages[-3:]

    print(f"修剪消息：从 {len(messages)} 条 → {len(new_messages)} 条")

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }

agent = create_agent(
    model=basic_model,
    middleware=[trim_messages, dynamic_deepseek_routing],
    checkpointer=InMemorySaver(),
)

In [29]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我是张三同学"}]},
    {"configurable": {"thread_id": "2"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请说一句鼓励我的话"}]},
    {"configurable": {"thread_id": "2"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "这句话有历史名人说过吗"}]},
    {"configurable": {"thread_id": "2"}},
)

===== 路由中间件被调用 =====
用户问题: 你好，我是张三同学
长度: 9
is_hard: False
最终模型: deepseek-chat
===== 路由中间件被调用 =====
用户问题: 请说一句鼓励我的话
长度: 9
is_hard: False
最终模型: deepseek-chat
修剪消息：从 5 条 → 4 条
===== 路由中间件被调用 =====
用户问题: 这句话有历史名人说过吗
长度: 11
is_hard: False
最终模型: deepseek-chat


In [30]:
response

{'messages': [HumanMessage(content='你好，我是张三同学', additional_kwargs={}, response_metadata={}, id='d82fcfbd-7a33-4a49-a1c1-e9a3194880a6'),
  HumanMessage(content='请说一句鼓励我的话', additional_kwargs={}, response_metadata={}, id='7f914e59-60ae-4c3b-b679-f4cd8f5eb413'),
  AIMessage(content='当然可以，张三同学！🌟\n\n**“每一个努力的身影，都在悄悄为未来的自己铺路——今天你流的汗，会变成明天闪光的底气。”** 💪\n\n别怕步子慢，只要在前进就好。无论遇到什么小挫折，记得你已经比昨天的自己更厉害了！需要能量的时候，我一直在这里陪你 🌱✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 116, 'total_tokens': 188, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 116}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'a05d39f1-3ae7-49ba-bb4e-cb8650b73bfd', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019efc96-5438-7ae2-ac50-bf94f7a546ed-0', tool

## 推理后确定是否要保存记忆

In [31]:
from langchain.agents.middleware import after_model

@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict | None:
    """模型调用后，删除最早的两条消息"""
    messages = state["messages"]

    if len(messages) > 4:
        removed = [m.id for m in messages[:2]]
        print(f"删除前两条消息，ID: {removed}")
        return {"messages": [RemoveMessage(id=m.id) for m in messages[:2]]}
    return None

agent = create_agent(
    model = basic_model,
    middleware=[trim_messages, dynamic_deepseek_routing, delete_old_messages],
    checkpointer=InMemorySaver(),
)

In [32]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我是李四"}]},
    {"configurable": {"thread_id": "3"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "默写一下静夜思"}]},
    {"configurable": {"thread_id": "3"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "今天真开心"}]},
    {"configurable": {"thread_id": "3"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "我的名字是什么？"}]},
    {"configurable": {"thread_id": "3"}},
)

print("\n最终回答：", response["messages"][-1].content)

===== 路由中间件被调用 =====
用户问题: 你好，我是李四
长度: 7
is_hard: False
最终模型: deepseek-chat
===== 路由中间件被调用 =====
用户问题: 默写一下静夜思
长度: 7
is_hard: False
最终模型: deepseek-chat
修剪消息：从 5 条 → 4 条
===== 路由中间件被调用 =====
用户问题: 今天真开心
长度: 5
is_hard: False
最终模型: deepseek-chat
删除前两条消息，ID: ['7e032f52-c7eb-43a4-a63e-078932032cda', '44ae715c-2059-4572-8b76-b86d8ab15fbc']
===== 路由中间件被调用 =====
用户问题: 我的名字是什么？
长度: 8
is_hard: False
最终模型: deepseek-chat
删除前两条消息，ID: ['lc_run--019efcae-6dc4-74d0-9cb4-e5e05dbf7782-0', 'b64b2333-7f38-4cd2-9e68-f5469fd32c30']

最终回答： 哈哈，看来你是在考我呀。根据咱们这段对话的上下文，我只知道你让我称呼你为“李四”（这是你上次提到的昵称），如果你有其他名字或者想让我换一种称呼，随时告诉我！


In [33]:
response

{'messages': [AIMessage(content='看到李四这么开心，真替你高兴！不如趁着这份心情，也来即兴对两句诗如何？\n\n既然你默写了李白的《静夜思》，那我也送你一首我现编的小诗，专为此刻的心情：\n\n**《喜日吟》**\n**白日放歌须纵酒，**\n**心头无事小神仙。**\n**莫问明朝风与雨，**\n**且将欢意付流年。**\n\n祝你天天都有今日这般好心情！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 51, 'total_tokens': 150, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 51}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'adc926da-f78d-4342-a135-5b2f12ede811', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019efcae-7137-7761-b250-54f3fd7d4083-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 99, 'total_tokens': 150, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
  HumanMessage(content='我的名字是什

输入输出以及中间件删除的流程
```
		H(3-after)
	AI(3-before)
		H(3-after)
			AI(4-after)
			H(4-after)
AI
H
AI
```

## 对用户记忆先总结后再进行推理，可实现无限上下文输入

In [55]:
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    # 主模型使用推理模型
    model=reasoner_model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            # 总结模型使用 chat 模型
            model=basic_model,
            trigger=("tokens", 2000),  # 超过 2000 token 触发记忆总结
            keep=("messages", 10),  # 总结记忆后保留最近10条原始消息
        )
    ],
    checkpointer=InMemorySaver(),
)

In [56]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我是李四"}]},
    {"configurable": {"thread_id": "4"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我今年 30 岁"}]},
    {"configurable": {"thread_id": "4"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我爱运动"}]},
    {"configurable": {"thread_id": "4"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我喜欢吃火锅"}]},
    {"configurable": {"thread_id": "4"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请以勇敢为主题讲一个 1000 字的故事"}]},
    {"configurable": {"thread_id": "4"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请以过年为主题讲一个 1000 字的故事"}]},
    {"configurable": {"thread_id": "4"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "请以过年为主题讲一个 1000 字的故事"}]},
    {"configurable": {"thread_id": "4"}},
)

In [57]:
for i in response['messages']:
    print(i, end = '\n\n')

content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is introducing themselves (李四, age 30). No specific task or goal has been stated yet; the session is in a preliminary greeting phase.\n\n## SUMMARY\n\n- User's name: 李四\n- User's age: 30 years old\n- No further requests, tasks, or decisions have been made.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\nAwait further instructions or requests from the user to determine the purpose of the session." additional_kwargs={'lc_source': 'summarization'} response_metadata={} id='91f71d09-830e-459b-bb0a-a6b6f1308a91'

content='好的，李四。30岁是一个很好的年纪，通常意味着更多的经验和更清晰的自我认知。你是想聊聊这个年龄阶段可能遇到的一些事情（比如职业、生活、心态），还是有什么具体的问题想探讨呢？' additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户说“你好，我今年30岁”。结合之前的对话历史，用户之前已经两次自我介绍为“李四”，我回应了“很高兴认识你”并开放了话题。现在用户补充了年龄信息。\n\n嗯，用户主动提供年龄，可能是在进一步介绍自己，希望我基于这个信息提供更个性化的回应或建议。30岁是一个常见的人生节点，用户可能正在经历一些与这个年龄相关的思考或困惑，比如职业发展、生活规划、家庭责任，或者对未来的焦虑。\n\n用户的深层需求可能不仅仅是告知年龄，而是希望得到一些与30岁这个年龄阶段相关的交流、建议或者共鸣。他/她

In [37]:
snapshot = agent.get_state(
    {
        "configurable":{
            "thread_id":"4"
        }
    }
)

In [38]:
for msg in snapshot.values["messages"]:
    print(type(msg))
    print(msg)

<class 'langchain_core.messages.human.HumanMessage'>
content='你好，我是李四' additional_kwargs={} response_metadata={} id='b7aec2f9-4c4b-4584-b6d3-ae6c9397a741'
<class 'langchain_core.messages.ai.AIMessage'>
content='你好，李四！我是DeepSeek，很高兴认识你。😊\n\n有什么我可以帮你的吗？无论是学习、工作、生活上的问题，还是只是想聊聊天，我都很乐意陪你聊聊！随便问吧～' additional_kwargs={'refusal': None, 'reasoning_content': '哦，用户直接介绍自己是李四。这是一个常见的自我介绍场景，需要友好回应并引导对话。可以用问候建立连接，同时开放对话话题。不需要特别处理，保持自然就好。考虑提供几个常见方向供选择，比如功能询问或闲聊，这样比较周全。'} response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 9, 'total_tokens': 114, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 57, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'db29e50a-6e91-4f27

In [52]:
agent = create_agent(
    model=reasoner_model,
    middleware=[
        SummarizationMiddleware(
            model=basic_model,
            trigger=("messages", 4),
            keep=("messages", 1),
        )
    ],
    checkpointer=InMemorySaver(),
)

In [53]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我是李四"}]},
    {"configurable": {"thread_id": "5"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我今年 30 岁"}]},
    {"configurable": {"thread_id": "5"}},
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "你好，我爱运动"}]},
    {"configurable": {"thread_id": "5"}},
)

In [54]:
for i in response['messages']:
    print(i, end = '\n\n')

content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\n用户自我介绍为李四，年龄30岁。助手与之建立初步认识，并表达愿意讨论职业发展、人生困惑、健康管理或生活感悟等话题。目前尚未明确用户的具体需求或问题。\n\n## SUMMARY\n\n- 用户姓名：李四\n- 用户年龄：30岁\n- 对话处于初始破冰阶段，尚未进入具体任务或问题的讨论。\n- 助手已表达开放态度，愿意倾听用户关于职业、人生、健康等方面的分享。\n\n## ARTIFACTS\n\n无。\n\n## NEXT STEPS\n\n- 等待用户提出具体的问题、目标或需要帮助的方向，以便进一步深入对话。' additional_kwargs={'lc_source': 'summarization'} response_metadata={} id='cea3d148-13a1-41d1-af95-0c815616dc93'

content='你好，我爱运动' additional_kwargs={} response_metadata={} id='e6ffd8a0-ba52-4727-9ae1-0b836b3eabb7'

content='你好，李四，很高兴再次与你交流！\n\n运动是个很好的习惯，尤其是在30岁这个阶段，保持身体的活力与健康既能提升能量，也有助于应对工作和生活的压力。你平时喜欢做什么类型的运动呢？是有规律的健身、跑步、游泳，还是更喜欢打球、骑行、瑜伽之类的项目？\n\n如果你在运动中遇到过什么困惑，比如怎么安排训练节奏、如何避免受伤、如何坚持下来，或者你想把运动习惯与职业状态、生活节奏更好地结合起来，我们都可以聊聊。我也很愿意听听你的运动故事，或者帮你梳理一些具体的目标。' additional_kwargs={'refusal': None, 'reasoning_content': '用户主动提及爱好运动，这是个很好的切入点可以建立连接。用户没有具体目标，但提到爱运动时通常期待要么获得技巧建议，要么想分享健身故事。我可以配合用户30岁年龄、职业发展有规划之类的背景，猜测他可能是想通过运动调节生活。  \n\n我需要先确认用户喜欢的运动类型，因为不同运动适合不同建议。跑步和

总结中间件其实本质上是before-model，只不过官方提供了已实现的方法。
注意其中的 trigger=("messages", 4)/trigger=("tokens", 2000)是两种不同的出发中间件方式

# 笔记

## memory
通过langgraph的checkpoint实现memory，以此实现多轮对话，而不是只能进行单轮对话。其中可以指定会话id.

## 中间件
可以利用中间件实现memory管理+模型路由
中间件有：before-model/after-model, wrap-model/wrap-tool，summary是官方实现的before-model中间件
使用中间件时，只需creat_agent时加上对应的middleware，同时最好加上checkpoint实现memory

### 中间件执行顺序，
Human
↓
before_model
↓
summary
↓
routing
↓
Model

生成:
tool_call(get_weather)

↓
wrap_tool_call
↓
Tool

返回:
ToolMessage

↓
before_model
↓
summary
↓
routing
↓
Model

生成:
最终回答

↓
after_model
↓
Checkpoint


